Build Deep Sequential Models & Training it from Scratch
=======================================================

# Setup

In [44]:
import torch


if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

'cuda'

In [45]:
from copy import deepcopy
import torch
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import kagglehub
import datasets
import torch.nn as nn
import tokenizers
import transformers
import pandas as pd
import torchmetrics
from torchmetrics.classification import MultilabelAveragePrecision

torch.manual_seed(42)

In [4]:
import gc

def del_vars(variable_names=[]):
    for name in variable_names:
        try:
            del globals()[name]
        except KeyError:
            pass  # ignore variables that have already been deleted
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

## Setup W&B for experiment tracking

In [5]:
import wandb

wandb.login(key="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")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: zaid_al_habbal (zaid_al_habbal-damascus-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

# Prepare the data 

## Get the data

In [25]:
data_path = kagglehub.dataset_download("julian3833/jigsaw-toxic-comment-classification-challenge",
                                        "train.csv",
                                        output_dir="../data"
                                        )
dataset = datasets.load_dataset("csv", data_files=data_path)
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate'],
        num_rows: 159571
    })
})

## Text Normalization

In [26]:
import re

# existing patterns
username_pattern = re.compile(r'\[\[user.*?\]\]|\[\[user talk.*?\]\]|user:\w+', re.IGNORECASE)
url_pattern = re.compile(r'http\S+|www\.\S+')  # Added URL pattern
whitespace_pattern = re.compile(r'\s+')
repeat_char_pattern = re.compile(r'(.)\1{2,}')

# wiki markup patterns
wiki_link_pattern = re.compile(r'\[\[(?:[^\]|]*\|)?([^\]]+)\]\]')
wiki_template_pattern = re.compile(r'\{\{.*?\}\}')
wiki_bold_italic_pattern = re.compile(r"'{2,5}")
wiki_heading_pattern = re.compile(r'=+\s*(.*?)\s*=+')

def preprocess_batch(batch):
    cleaned_texts = []

    for text in batch["comment_text"]:
        # Handle None or empty input
        if not text or not text.strip():
            cleaned_texts.append("[UNK]")  # Placeholder for empty texts
            continue
        
        text = text.lower()

        # remove usernames
        text = username_pattern.sub(" ", text)
        
        # Replace URLs with [UNK]
        text = url_pattern.sub("[UNK]", text)

        # convert [[link|text]] -> text
        text = wiki_link_pattern.sub(r"\1", text)

        # remove templates {{...}}
        text = wiki_template_pattern.sub(" ", text)

        # remove bold/italic markup
        text = wiki_bold_italic_pattern.sub("", text)

        # remove headings
        text = wiki_heading_pattern.sub(r"\1", text)

        # normalize stretched characters
        text = repeat_char_pattern.sub(r"\1\1", text)

        # normalize whitespace
        text = whitespace_pattern.sub(" ", text).strip()

        # ✅ Ensure text is not empty after cleaning
        if not text:
            text = "[UNK]"  # Fallback for texts that become empty

        cleaned_texts.append(text)

    batch["comment_text"] = cleaned_texts
    return batch

In [27]:
dataset = dataset["train"].map(preprocess_batch, batched=True, num_proc=4)

Map (num_proc=4):   0%|          | 0/159571 [00:00<?, ? examples/s]

## Tokenization

### Train a BBPE from scratch

#### Prepare the comments for training

In [28]:
train = dataset.train_test_split(test_size=0.2, seed=42)["train"]
comments = [comment['comment_text'] for comment in train]
comments[:2]

["missing champions this article should have all the tna world heavyweight champions, from ken shamrock. weird that it doesn't.",
 '":this article is essentially puffery, written by hazare or one of his fanatics. ""much-acclaimed popular indian social activist"" reads like one of his propaganda articles. the only difference is the change from ""well-acclaimed"" which, in typical indian english style (i.e. poor english), he uses. "']

#### Train the tokenizer

In [29]:
special_tokens = ['<pad>', '<unk>']


bbpe_model = tokenizers.models.BPE(unk_token="<unk>")
bbpe_tokenizer = tokenizers.Tokenizer(bbpe_model)
bbpe_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.ByteLevel()
bbpe_trainer = tokenizers.trainers.BpeTrainer(vocab_size=30_000,
                                              special_tokens=special_tokens)
bbpe_tokenizer.train_from_iterator(comments, bbpe_trainer)

bbpe_tokenizer.enable_padding(
    pad_id=0,
    pad_token="<pad>"
)

#### Wrap our tokenizer in a HuggingFace PreTrainedTokenizerFast for easy integration

In [11]:
tokenizer = transformers.PreTrainedTokenizerFast(
    tokenizer_object=bbpe_tokenizer
)

#### Testing our tokenizer

In [30]:
encodings = tokenizer(comments[:2], truncation=True, max_length=256)

encodings

{'input_ids': [[3100, 5915, 258, 282, 438, 284, 377, 186, 15650, 1064, 19454, 5915, 13, 362, 4475, 11940, 9484, 15, 5616, 232, 231, 943, 309, 15], [1850, 2226, 282, 226, 5442, 22239, 13, 1412, 358, 5556, 497, 297, 414, 217, 486, 17125, 15, 288, 15331, 14, 4822, 4803, 193, 2951, 3040, 2414, 10518, 275, 4573, 405, 414, 217, 486, 3243, 548, 15, 186, 527, 2620, 226, 186, 1074, 362, 288, 3330, 14, 4822, 4803, 193, 275, 457, 13, 216, 5563, 3040, 1220, 1307, 271, 48, 15, 44, 15, 2592, 1220, 778, 299, 2740, 15, 234]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}

In [32]:
encodings["input_ids"]

[[3100,
  5915,
  258,
  282,
  438,
  284,
  377,
  186,
  15650,
  1064,
  19454,
  5915,
  13,
  362,
  4475,
  11940,
  9484,
  15,
  5616,
  232,
  231,
  943,
  309,
  15],
 [1850,
  2226,
  282,
  226,
  5442,
  22239,
  13,
  1412,
  358,
  5556,
  497,
  297,
  414,
  217,
  486,
  17125,
  15,
  288,
  15331,
  14,
  4822,
  4803,
  193,
  2951,
  3040,
  2414,
  10518,
  275,
  4573,
  405,
  414,
  217,
  486,
  3243,
  548,
  15,
  186,
  527,
  2620,
  226,
  186,
  1074,
  362,
  288,
  3330,
  14,
  4822,
  4803,
  193,
  275,
  457,
  13,
  216,
  5563,
  3040,
  1220,
  1307,
  271,
  48,
  15,
  44,
  15,
  2592,
  1220,
  778,
  299,
  2740,
  15,
  234]]

#### Use our tokenizer

In [33]:
def tokenize_batch(batch):
    return tokenizer(
        batch["comment_text"],
        truncation=True,
        max_length=256,
        padding=False
    )

In [34]:
dataset = dataset.map(
    tokenize_batch,
    batched=True,
    num_proc=4
)
dataset = dataset.remove_columns(["comment_text", "id"])

dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "toxic",
        "severe_toxic",
        "obscene",
        "threat",
        "insult",
        "identity_hate"
    ]
)

Map (num_proc=4):   0%|          | 0/159571 [00:00<?, ? examples/s]

## Split data

In [35]:
from transformers import DataCollatorWithPadding


LABEL_COLUMNS = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

# Create a data collator that handles padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def collate_fn(batch):
    # Extract text features for padding
    text_features = [{
        "input_ids": item["input_ids"],
        "attention_mask": item["attention_mask"]
    } for item in batch]
    
    # Pad the batch
    padded = data_collator(text_features)
    
    # Extract labels
    labels = torch.stack([
        torch.tensor([item[label] for label in LABEL_COLUMNS], dtype=torch.float32)
        for item in batch
    ])
    
    X = {
        "input_ids": padded["input_ids"],
        "attention_mask": padded["attention_mask"]
    }
    
    return X, labels

In [36]:
split = dataset.train_test_split(test_size=0.2, seed=42)
train_set = split["train"]
test_valid_set = split["test"]
split = test_valid_set.train_test_split(test_size=0.5, seed=42)
valid_set = split["train"]
test_set = split["test"]

### Splits statistics

In [37]:
print(f"Train Shape: {train_set.shape}")
print(f"Valid Shape: {valid_set.shape}")
print(f"Test Shape: {test_set.shape}")
print("------------------------------------------------")

def calculate_label_statistics(dataset, split_name):
    """Calculate label statistics for a given dataset split."""
    
    # Convert to pandas DataFrame for easier manipulation
    df = dataset.to_pandas()
    
    # Extract label columns
    label_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
    labels = df[label_cols]
    
    # Calculate statistics
    stats = {}
    
    # 1. Number of rows without any toxic label (all labels are 0)
    stats['no_toxic_labels'] = (labels.sum(axis=1) == 0).sum()
    
    # 2. Number of rows with at least one toxic label
    stats['at_least_one_toxic'] = (labels.sum(axis=1) > 0).sum()
    
    # 3. For each label: number of rows where it takes value 1
    for col in label_cols:
        stats[f'{col}_count'] = labels[col].sum()
    
    # Add total rows for reference
    stats['total_rows'] = len(df)
    
    return stats

# Calculate statistics for each split
train_stats = calculate_label_statistics(train_set, "Train")
valid_stats = calculate_label_statistics(valid_set, "Valid")
test_stats = calculate_label_statistics(test_set, "Test")

# Create a summary DataFrame
summary_stats = pd.DataFrame({
    'Train': train_stats,
    'Valid': valid_stats, 
    'Test': test_stats
}).T

# Display the results
print("=== Label Statistics Summary ===")
print(summary_stats)

# Calculate percentages for better understanding
print("\n=== Label Distribution Percentages ===")
percentage_stats = summary_stats.copy()
for split in ['Train', 'Valid', 'Test']:
    total = summary_stats.loc[split, 'total_rows']
    percentage_stats.loc[split, 'no_toxic_labels_pct'] = (summary_stats.loc[split, 'no_toxic_labels'] / total) * 100
    percentage_stats.loc[split, 'at_least_one_toxic_pct'] = (summary_stats.loc[split, 'at_least_one_toxic'] / total) * 100
    
    for col in ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']:
        percentage_stats.loc[split, f'{col}_pct'] = (summary_stats.loc[split, f'{col}_count'] / total) * 100

# Display percentage columns
percentage_cols = ['no_toxic_labels_pct', 'at_least_one_toxic_pct'] + [f'{col}_pct' for col in ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']]
print(percentage_stats[percentage_cols].round(2))

VOCAB_SIZE = len(tokenizer)
print(f"Vocabulary size: {VOCAB_SIZE}")


Train Shape: (127656, 8)
Valid Shape: (15957, 8)
Test Shape: (15958, 8)
------------------------------------------------
=== Label Statistics Summary ===
       no_toxic_labels  at_least_one_toxic  toxic_count  severe_toxic_count  \
Train           114633               13023        12276                1296   
Valid            14395                1562         1474                 145   
Test             14318                1640         1544                 154   

       obscene_count  threat_count  insult_count  identity_hate_count  \
Train           6773           391          6307                 1137   
Valid            851            45           781                  132   
Test             825            42           789                  136   

       total_rows  
Train      127656  
Valid       15957  
Test        15958  

=== Label Distribution Percentages ===
       no_toxic_labels_pct  at_least_one_toxic_pct  toxic_pct  \
Train                89.80                   10.20 

### Positive class weights

In [59]:
# compute per-label positive/negative counts on the training split
df_train = train_set.to_pandas()
label_counts = df_train[LABEL_COLUMNS].sum()
total_examples = len(df_train)

neg_counts = total_examples - label_counts

# pos_weight is commonly used with BCEWithLogitsLoss for multi-label tasks
pos_weight = (neg_counts / (label_counts + 1e-9)).astype("float32")

print("Label counts (pos):")
print(label_counts)
print("\nPos weights (neg/pos):")
print(pos_weight)

# store as torch tensor for passing to loss
pos_weight_tensor = torch.tensor(pos_weight.values, dtype=torch.float32).to(device)
pos_weight_tensor

Label counts (pos):
toxic            12276
severe_toxic      1296
obscene           6773
threat             391
insult            6307
identity_hate     1137
dtype: int64

Pos weights (neg/pos):
toxic              9.398827
severe_toxic      97.500000
obscene           17.847778
threat           325.485931
insult            19.240368
identity_hate    111.274406
dtype: float32


tensor([  9.3988,  97.5000,  17.8478, 325.4859,  19.2404, 111.2744],
       device='cuda:0')

## Load the datasets into PyTorch DataLoaders

In [38]:
BATCH_SIZE = 256

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_set, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

In [39]:
# Add after creating dataloaders
sample_batch = next(iter(train_loader))
X_sample, y_sample = sample_batch

print("Input shapes:")
print(f"  input_ids: {X_sample['input_ids'].shape}")
print(f"  attention_mask: {X_sample['attention_mask'].shape}")
print(f"  labels: {y_sample.shape}")
print(f"  labels dtype: {y_sample.dtype}")

assert X_sample['input_ids'].shape[0] == BATCH_SIZE
assert y_sample.shape == (BATCH_SIZE, 6)
assert y_sample.dtype == torch.float32

Input shapes:
  input_ids: torch.Size([256, 256])
  attention_mask: torch.Size([256, 256])
  labels: torch.Size([256, 6])
  labels dtype: torch.float32


# Train & Eval functions

In [40]:
import torchmetrics
from torchmetrics.classification import MultilabelAveragePrecision

def evaluate_tm(model, data_loader, metric):
    """Evaluate model on a dataset."""
    
    model.eval()
    metric.reset()

    with torch.no_grad():
        for X_batch, y_batch in data_loader:

            X_batch = {k: v.to(device) for k, v in X_batch.items()}
            y_batch = y_batch.to(device)

            logits = model(X_batch)
            probs = torch.sigmoid(logits)

            metric.update(probs.detach(), y_batch.int())

    return metric.compute()


def train(
    model,
    optimizer,
    loss_fn,
    train_metric,
    val_metric,
    train_loader,
    valid_loader,
    n_epochs,
    patience=2,
    factor=0.5,
    max_grad_norm=1.0,
    checkpoint_path="best_model.pth",
):

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        patience=patience,
        factor=factor
    )

    history = {
        "train_losses": [],
        "train_metrics": [],
        "valid_metrics": [],
    }

    best_val_metric = 0.0

    for epoch in range(n_epochs):

        model.train()
        train_metric.reset()

        total_loss = 0.0

        for index, (X_batch, y_batch) in enumerate(train_loader):

            X_batch = {k: v.to(device) for k, v in X_batch.items()}
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            logits = model(X_batch)

            loss = loss_fn(logits, y_batch)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

            optimizer.step()

            total_loss += loss.item()

            probs = torch.sigmoid(logits)

            train_metric.update(
                probs.detach(),
                y_batch.int()
            )

            print(
                f"\rBatch {index+1}/{len(train_loader)} "
                f"loss={total_loss/(index+1):.4f}",
                end=""
            )

        # ----- Epoch metrics -----

        train_metric_value = train_metric.compute().item()
        val_metric_value = evaluate_tm(model, valid_loader, val_metric).item()

        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(train_metric_value)
        history["valid_metrics"].append(val_metric_value)

        scheduler.step(val_metric_value)

        if val_metric_value > best_val_metric:

            best_val_metric = val_metric_value

            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "val_metric": val_metric_value,
                    "history": history,
                },
                checkpoint_path,
            )

        print(
            f"\rEpoch {epoch+1}/{n_epochs}, "
            f"train loss: {history['train_losses'][-1]:.4f}, "
            f"train metric: {train_metric_value:.2%}, "
            f"valid metric: {val_metric_value:.2%} "
            f"{'[BEST]' if val_metric_value == best_val_metric else ''}"
        )

        wandb.log({
            "train/loss": history['train_losses'][-1],
            "train/pr_auc": train_metric_value,
            "val/pr_auc": val_metric_value,
            "epoch": epoch+1/n_epochs
        })

    return history

# Model Building, Training & Evaluation

## Simple Stacked LSTM

In [41]:
class SimpleStackedLSTMModel(nn.Module):
    def __init__(self, vocab_size=30522, embed_dim=256, n_layers=2,
                 hidden_dim=128, pad_id=0, dropout=0.2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=n_layers,
                            batch_first=True, dropout=dropout)
        self.output = nn.Linear(hidden_dim, 6)

    def forward(self, encodings):
        embeddings = self.embed(encodings["input_ids"])
        lengths = encodings["attention_mask"].sum(dim=1)                      
        packed = pack_padded_sequence(embeddings, lengths=lengths.cpu(),      
                                      batch_first=True, enforce_sorted=False) 
        _outputs, (h_n, c_n) = self.lstm(packed) 
        return self.output(h_n[-1])

### Base Architecture & Hyperparameters

In [42]:
base_config = {
    # Model architecture
    "model_type": "LSTM",
    "embed_dim": 256,
    "hidden_dim": 128,
    "num_layers": 2,
    "dropout": 0.2,
    
    # Training
    "learning_rate": 0.001,
    "batch_size": 256,
    "epochs": 10,
    "optimizer": "NAdam",
}

### Config_1 Training

In [46]:
# Setup New Config
config = base_config

model = SimpleStackedLSTMModel(
    vocab_size=len(tokenizer),
    embed_dim=config["embed_dim"],
    n_layers=config["num_layers"],
    hidden_dim=config["hidden_dim"],
    dropout=config["dropout"]
).to(device)

# Training configuration
n_epochs = config["epochs"]
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.NAdam(model.parameters(), lr=config["learning_rate"])

# Metric (move to device!)
macro_pr_auc = MultilabelAveragePrecision(
    num_labels=6,
    average="macro"
).to(device)

# Train
with wandb.init(project="nontoxic-world",
                name="from-scratch-simple-stacked-lstm-config-1",
                config=config,
                notes="base config for simple stacked lstm model"
               ):
    history = train(
        model, 
        optimizer, 
        xentropy, 
        macro_pr_auc,
        macro_pr_auc,
        train_loader, 
        valid_loader, 
        n_epochs,
        max_grad_norm=1.0,
        checkpoint_path='simple_stacked_lstm_model_best.pth'
    )

Epoch 1/10, train loss: 0.0917, train metric: 33.04%, valid metric: 49.39% [BEST]
Epoch 2/10, train loss: 0.0508, train metric: 51.18%, valid metric: 52.82% [BEST]
Epoch 3/10, train loss: 0.0438, train metric: 54.42%, valid metric: 54.10% [BEST]
Epoch 4/10, train loss: 0.0389, train metric: 56.85%, valid metric: 54.98% [BEST]
Epoch 5/10, train loss: 0.0344, train metric: 60.63%, valid metric: 56.83% [BEST]
Epoch 6/10, train loss: 0.0302, train metric: 65.26%, valid metric: 57.73% [BEST]
Epoch 7/10, train loss: 0.0264, train metric: 70.86%, valid metric: 59.57% [BEST]
Epoch 8/10, train loss: 0.0232, train metric: 76.53%, valid metric: 61.00% [BEST]
Epoch 9/10, train loss: 0.0200, train metric: 81.34%, valid metric: 59.91% 
Epoch 10/10, train loss: 0.0172, train metric: 84.86%, valid metric: 61.35% [BEST]


epoch,▁▂▃▃▄▅▆▆▇█
train/loss,█▄▄▃▃▂▂▂▁▁
train/pr_auc,▁▃▄▄▅▅▆▇██
val/pr_auc,▁▃▄▄▅▆▇█▇█
epoch,9.1
train/loss,0.01717
train/pr_auc,0.8486
val/pr_auc,0.61354


**Ammmmm it seems like our model is overfiting the train data**

### Config_2 Training

In [47]:
# Setup New Config
config = deepcopy(base_config)
config["hidden_dim"] = 64

model = SimpleStackedLSTMModel(
    vocab_size=len(tokenizer),
    embed_dim=config["embed_dim"],
    n_layers=config["num_layers"],
    hidden_dim=config["hidden_dim"],
    dropout=config["dropout"]
).to(device)


# Training configuration
n_epochs = config["epochs"]
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.NAdam(model.parameters(), lr=config["learning_rate"])

# Metric (move to device!)
macro_pr_auc = MultilabelAveragePrecision(
    num_labels=6,
    average="macro"
).to(device)

# Train
with wandb.init(project="nontoxic-world",
                name="from-scratch-simple-stacked-lstm-config-2",
                config=config,
                notes="changed hidden_dim to 64"
               ):
    
    history = train(
        model, 
        optimizer, 
        xentropy, 
        macro_pr_auc,
        macro_pr_auc,
        train_loader, 
        valid_loader, 
        n_epochs,
        max_grad_norm=1.0,
        checkpoint_path='simple_stacked_lstm_model_best.pth'
    )

Epoch 1/10, train loss: 0.1080, train metric: 27.95%, valid metric: 48.71% [BEST]
Epoch 2/10, train loss: 0.0527, train metric: 50.26%, valid metric: 51.56% [BEST]
Epoch 3/10, train loss: 0.0457, train metric: 51.94%, valid metric: 51.24% 
Epoch 4/10, train loss: 0.0406, train metric: 54.78%, valid metric: 52.03% [BEST]
Epoch 5/10, train loss: 0.0367, train metric: 56.99%, valid metric: 53.89% [BEST]
Epoch 6/10, train loss: 0.0338, train metric: 59.40%, valid metric: 54.10% [BEST]
Epoch 7/10, train loss: 0.0309, train metric: 62.78%, valid metric: 55.29% [BEST]
Epoch 8/10, train loss: 0.0280, train metric: 66.83%, valid metric: 54.60% 
Epoch 9/10, train loss: 0.0256, train metric: 69.98%, valid metric: 55.57% [BEST]
Epoch 10/10, train loss: 0.0233, train metric: 72.87%, valid metric: 55.85% [BEST]


epoch,▁▂▃▃▄▅▆▆▇█
train/loss,█▃▃▂▂▂▂▁▁▁
train/pr_auc,▁▄▅▅▆▆▆▇██
val/pr_auc,▁▄▃▄▆▆▇▇██
epoch,9.1
train/loss,0.02331
train/pr_auc,0.72868
val/pr_auc,0.55848


### Config_3 Training

In [48]:
# Setup New Config
config = deepcopy(base_config)
config["hidden_dim"] = 64
config["dropout"] = 0.3

model = SimpleStackedLSTMModel(
    vocab_size=len(tokenizer),
    embed_dim=config["embed_dim"],
    n_layers=config["num_layers"],
    hidden_dim=config["hidden_dim"],
    dropout=config["dropout"]
).to(device)


# Training configuration
n_epochs = config["epochs"]
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.NAdam(model.parameters(), lr=config["learning_rate"])

# Metric (move to device!)
macro_pr_auc = MultilabelAveragePrecision(
    num_labels=6,
    average="macro"
).to(device)

# Train
with wandb.init(project="nontoxic-world",
                name="from-scratch-simple-stacked-lstm-config-3",
                config=config,
                notes="changed hidden_dim + dropout values"
               ):
    history = train(
        model, 
        optimizer, 
        xentropy, 
        macro_pr_auc,
        macro_pr_auc,
        train_loader, 
        valid_loader, 
        n_epochs,
        max_grad_norm=1.0,
        checkpoint_path='simple_stacked_lstm_model_best.pth'
    )

Epoch 1/10, train loss: 0.1089, train metric: 26.74%, valid metric: 47.08% [BEST]
Epoch 2/10, train loss: 0.0549, train metric: 49.48%, valid metric: 50.75% [BEST]
Epoch 3/10, train loss: 0.0465, train metric: 52.58%, valid metric: 52.33% [BEST]
Epoch 4/10, train loss: 0.0420, train metric: 54.82%, valid metric: 52.14% 
Epoch 5/10, train loss: 0.0388, train metric: 56.33%, valid metric: 52.60% [BEST]
Epoch 6/10, train loss: 0.0355, train metric: 58.41%, valid metric: 54.06% [BEST]
Epoch 7/10, train loss: 0.0324, train metric: 61.00%, valid metric: 55.05% [BEST]
Epoch 8/10, train loss: 0.0295, train metric: 64.69%, valid metric: 56.05% [BEST]
Epoch 9/10, train loss: 0.0269, train metric: 68.41%, valid metric: 57.41% [BEST]
Epoch 10/10, train loss: 0.0248, train metric: 71.37%, valid metric: 58.02% [BEST]


epoch,▁▂▃▃▄▅▆▆▇█
train/loss,█▄▃▂▂▂▂▁▁▁
train/pr_auc,▁▅▅▅▆▆▆▇██
val/pr_auc,▁▃▄▄▅▅▆▇██
epoch,9.1
train/loss,0.02482
train/pr_auc,0.71372
val/pr_auc,0.58017


### Config_4 Training

In [49]:
# Setup New Config
config = deepcopy(base_config)
config["hidden_dim"] = 32

model = SimpleStackedLSTMModel(
    vocab_size=len(tokenizer),
    embed_dim=config["embed_dim"],
    n_layers=config["num_layers"],
    hidden_dim=config["hidden_dim"],
    dropout=config["dropout"]
).to(device)


# Training configuration
n_epochs = config["epochs"]
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.NAdam(model.parameters(), lr=config["learning_rate"])

# Metric (move to device!)
macro_pr_auc = MultilabelAveragePrecision(
    num_labels=6,
    average="macro"
).to(device)

# Train
with wandb.init(project="nontoxic-world",
                name="from-scratch-simple-stacked-lstm-config-4",
                config=config,
                notes="changed hidden_dim to 32"
               ):

    history = train(
        model, 
        optimizer, 
        xentropy, 
        macro_pr_auc,
        macro_pr_auc,
        train_loader, 
        valid_loader, 
        n_epochs,
        max_grad_norm=1.0,
        checkpoint_path='simple_stacked_lstm_model_best.pth'
    )

Epoch 1/10, train loss: 0.1339, train metric: 22.19%, valid metric: 46.63% [BEST]
Epoch 2/10, train loss: 0.0570, train metric: 49.13%, valid metric: 51.11% [BEST]
Epoch 3/10, train loss: 0.0476, train metric: 52.66%, valid metric: 52.02% [BEST]
Epoch 4/10, train loss: 0.0426, train metric: 54.32%, valid metric: 52.50% [BEST]
Epoch 5/10, train loss: 0.0393, train metric: 55.80%, valid metric: 53.09% [BEST]
Epoch 6/10, train loss: 0.0363, train metric: 56.89%, valid metric: 53.32% [BEST]
Epoch 7/10, train loss: 0.0338, train metric: 58.33%, valid metric: 53.74% [BEST]
Epoch 8/10, train loss: 0.0316, train metric: 60.56%, valid metric: 53.66% 
Epoch 9/10, train loss: 0.0303, train metric: 61.44%, valid metric: 53.51% 
Epoch 10/10, train loss: 0.0285, train metric: 63.87%, valid metric: 53.35% 


epoch,▁▂▃▃▄▅▆▆▇█
train/loss,█▃▂▂▂▂▁▁▁▁
train/pr_auc,▁▆▆▆▇▇▇▇██
val/pr_auc,▁▅▆▇▇█████
epoch,9.1
train/loss,0.02852
train/pr_auc,0.6387
val/pr_auc,0.53353


In [51]:
# Train it more 10 epochs
config["epochs"] = 20

with wandb.init(project="nontoxic-world",
                id="nzdtqh0a",
                resume="allow"
               ):
    
    history = train(
        model, 
        optimizer, 
        xentropy, 
        macro_pr_auc,
        macro_pr_auc,
        train_loader, 
        valid_loader, 
        n_epochs,
        max_grad_norm=1.0,
        checkpoint_path='simple_stacked_lstm_model_best.pth'
    )

Epoch 1/10, train loss: 0.0262, train metric: 66.79%, valid metric: 53.28% [BEST]
Epoch 2/10, train loss: 0.0248, train metric: 68.19%, valid metric: 53.51% [BEST]
Epoch 3/10, train loss: 0.0239, train metric: 69.66%, valid metric: 53.45% 
Epoch 4/10, train loss: 0.0229, train metric: 71.71%, valid metric: 52.82% 
Epoch 5/10, train loss: 0.0222, train metric: 72.69%, valid metric: 53.43% 
Epoch 6/10, train loss: 0.0207, train metric: 75.26%, valid metric: 52.90% 
Epoch 7/10, train loss: 0.0200, train metric: 76.13%, valid metric: 52.84% 
Epoch 8/10, train loss: 0.0195, train metric: 76.98%, valid metric: 52.97% 
Epoch 9/10, train loss: 0.0188, train metric: 78.13%, valid metric: 52.92% 
Epoch 10/10, train loss: 0.0184, train metric: 78.83%, valid metric: 52.78% 


epoch,▁▂▃▃▄▅▆▆▇█
train/loss,█▇▆▅▄▃▂▂▁▁
train/pr_auc,▁▂▃▄▄▆▆▇██
val/pr_auc,▆█▇▁▇▂▂▃▂▁
epoch,9.1
train/loss,0.01842
train/pr_auc,0.78827
val/pr_auc,0.52776


### Config_5 Training

In [57]:
# Setup New Config
config = deepcopy(base_config)
config["optimizer"] = "AdamW"

model = SimpleStackedLSTMModel(
    vocab_size=len(tokenizer),
    embed_dim=config["embed_dim"],
    n_layers=config["num_layers"],
    hidden_dim=config["hidden_dim"],
    dropout=config["dropout"]
).to(device)


# Training configuration
n_epochs = config["epochs"]
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"])

# Metric (move to device!)
macro_pr_auc = MultilabelAveragePrecision(
    num_labels=6,
    average="macro"
).to(device)

# Train
with wandb.init(project="nontoxic-world",
                name="from-scratch-simple-stacked-lstm-config-5",
                config=config,
                notes="changed optimizer to AdamW"
               ):
    
    history = train(
        model, 
        optimizer, 
        xentropy, 
        macro_pr_auc,
        macro_pr_auc,
        train_loader, 
        valid_loader, 
        n_epochs,
        max_grad_norm=1.0,
        checkpoint_path='simple_stacked_lstm_model_best.pth'
    )

Epoch 1/10, train loss: 0.1005, train metric: 29.31%, valid metric: 49.74% [BEST]
Epoch 2/10, train loss: 0.0516, train metric: 50.82%, valid metric: 52.35% [BEST]
Epoch 3/10, train loss: 0.0434, train metric: 54.31%, valid metric: 53.60% [BEST]
Epoch 4/10, train loss: 0.0389, train metric: 56.58%, valid metric: 54.39% [BEST]
Epoch 5/10, train loss: 0.0346, train metric: 59.26%, valid metric: 55.43% [BEST]
Epoch 6/10, train loss: 0.0306, train metric: 63.19%, valid metric: 55.90% [BEST]
Epoch 7/10, train loss: 0.0272, train metric: 67.10%, valid metric: 57.45% [BEST]
Epoch 8/10, train loss: 0.0245, train metric: 71.18%, valid metric: 57.92% [BEST]
Epoch 9/10, train loss: 0.0217, train metric: 75.68%, valid metric: 58.82% [BEST]
Epoch 10/10, train loss: 0.0191, train metric: 81.07%, valid metric: 58.96% [BEST]


epoch,▁▂▃▃▄▅▆▆▇█
train/loss,█▄▃▃▂▂▂▁▁▁
train/pr_auc,▁▄▄▅▅▆▆▇▇█
val/pr_auc,▁▃▄▅▅▆▇▇██
epoch,9.1
train/loss,0.01905
train/pr_auc,0.81071
val/pr_auc,0.58959


In [ ]:
# Setup New Config
config = deepcopy(base_config)

model = SimpleStackedLSTMModel(
    vocab_size=len(tokenizer),
    embed_dim=config["embed_dim"],
    n_layers=config["num_layers"],
    hidden_dim=config["hidden_dim"],
    dropout=config["dropout"]
).to(device)


# Training configuration
n_epochs = config["epochs"]
xentropy = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
optimizer = torch.optim.NAdam(model.parameters(), lr=config["learning_rate"])

# Metric (move to device!)
macro_pr_auc = MultilabelAveragePrecision(
    num_labels=6,
    average="macro"
).to(device)

# Train
with wandb.init(project="nontoxic-world",
                name="from-scratch-simple-stacked-lstm-config-6",
                config=config,
                notes="use pos class weights for the loss function"
               ):
    
    history = train(
        model, 
        optimizer, 
        xentropy, 
        macro_pr_auc,
        macro_pr_auc,
        train_loader, 
        valid_loader, 
        n_epochs,
        max_grad_norm=1.0,
        checkpoint_path='simple_stacked_lstm_model_best.pth'
    )

Batch 50/499 loss=1.1133

#### Evaluate Each Label seperatly

In [55]:
val_metric = MultilabelAveragePrecision(
    num_labels=6,
    average=None
).to(device)

results = evaluate_tm(model, valid_loader, val_metric)
print(results)

tensor([0.8403, 0.3738, 0.8691, 0.1041, 0.7294, 0.1936], device='cuda:0')


In [56]:
train_metric = MultilabelAveragePrecision(
    num_labels=6,
    average=None
).to(device)

results = evaluate_tm(model, train_loader, train_metric)
print(results)

tensor([0.9792, 0.6479, 0.9645, 0.2165, 0.9054, 0.4433], device='cuda:0')
